### Classic Transformer

复现《Attention is All You Need》（Vaswani et al., 2017）。

**第一步：缩放点积注意力（Scaled Dot-Product Attention）**

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [5]:
import math
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


class ScaledDotProductAttention(nn.Module):
    """
    缩放点积注意力（论文 Section 3.2.1）。
    Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V
    其中 d_k 为每个 head 上 query/key 的特征维，缩放避免点积过大导致 softmax 梯度消失。
    """

    def __init__(self, dropout: float = 0.0) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        参数
        ----
        q : Tensor
            Query，形状 [batch_size, *, seq_len_q, d_k]。
            * 可为 num_heads（多头时）或其它可广播前缀；下面以单头写法说明。
        k : Tensor
            Key，形状 [batch_size, *, seq_len_k, d_k]。
        v : Tensor
            Value，形状 [batch_size, *, seq_len_k, d_v]；论文里常取 d_v = d_k。
        mask : Tensor, optional
            与 scores 广播相容即可，常见为：
            - bool：True 表示该位置 **禁止** 参与注意力（padding / 因果遮挡）；
            - float：直接 **加到** scores 上（例如 0 保留，-inf 屏蔽）。

        返回
        ----
        output : Tensor
            [batch_size, *, seq_len_q, d_v]。
        attn_weights : Tensor
            [batch_size, *, seq_len_q, seq_len_k]，softmax 后的注意力权重（dropout 前保存的
            版本若需用于可视化，可自行改在 dropout 之前返回一份拷贝）。
        """
        # d_k：最后一维，即每个位置 query/key 的通道数
        d_k = q.size(-1)

        # 点积 scores: Q @ K^T
        # q: [batch_size, *, seq_len_q, d_k]
        # k: [batch_size, *, seq_len_k, d_k] -> k^T 在最后两维: [batch_size, *, d_k, seq_len_k]
        # scores: [batch_size, *, seq_len_q, seq_len_k]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            if mask.dtype == torch.bool:
                scores = scores.masked_fill(mask, float("-inf"))
            else:
                scores = scores + mask

        # 在最后一维（key 序列维）上做 softmax，每行 query 位置得到对 key 的分布
        # attn_weights: [batch_size, *, seq_len_q, seq_len_k]
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 对 V 加权求和
        # attn_weights: [batch_size, *, seq_len_q, seq_len_k]
        # v: [batch_size, *, seq_len_k, d_v]
        # output: [batch_size, *, seq_len_q, d_v]
        output = torch.matmul(attn_weights, v)
        return output, attn_weights

In [6]:
# 形状自检（单头示例）
batch_size, seq_len_q, seq_len_k, d_k, d_v = 2, 4, 5, 8, 8
sdpa = ScaledDotProductAttention(dropout=0.1)
sdpa.eval()

Q = torch.randn(batch_size, seq_len_q, d_k)
K = torch.randn(batch_size, seq_len_k, d_k)
V = torch.randn(batch_size, seq_len_k, d_v)

out, w = sdpa(Q, K, V)
assert out.shape == (batch_size, seq_len_q, d_v)
assert w.shape == (batch_size, seq_len_q, seq_len_k)
assert torch.allclose(w.sum(dim=-1), torch.ones(batch_size, seq_len_q), atol=1e-5)
print("out:", out.shape, "attn_weights:", w.shape)

out: torch.Size([2, 4, 8]) attn_weights: torch.Size([2, 4, 5])


**第二步：多头注意力（Multi-Head Attention）**

论文 Section 3.2.2：$\mathrm{MultiHead}(Q,K,V)=\mathrm{Concat}(\mathrm{head}_1,\ldots,\mathrm{head}_h)W^O$，其中每个 $\mathrm{head}_i=\mathrm{Attention}(QW_i^Q,KW_i^K,VW_i^V)$。

- **Encoder 自注意力**：`query = key = value` 为源序列同一表示；mask 常为 padding（key 维屏蔽）。
- **Decoder 自注意力**：三者同为解码端序列；需叠加 **因果** mask（只看过去）。
- **Encoder–Decoder 交叉注意力**：`query` 来自解码器，`key` / `value` 来自编码器输出；序列长度可不同（$L_q$ 与 $L_k$）。

以下 `forward(query, key, value, mask=None)` 通过传入不同张量覆盖上述三种用法。

In [7]:
class MultiHeadAttention(nn.Module):
    """
    多头注意力（论文 Section 3.2.2）。
    在 d_model 维上做 Q/K/V 线性投影，拆成 h 个头并行缩放点积注意力，拼接后再经 W^O。
    """

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) 必须能被 num_heads ({num_heads}) 整除")
        self.d_model = d_model
        self.num_heads = num_heads
        # 每个头上的维度 d_k = d_v = d_model / h
        self.d_k = d_model // num_heads

        # 输入 -> Q, K, V：各为 d_model -> d_model，内含 h 个 d_k 维子空间
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention(dropout=dropout)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        参数
        ----
        query : Tensor
            [batch_size, seq_len_q, d_model]。Decoder 侧自注意力 / 交叉注意力中均为解码 query。
        key, value : Tensor
            [batch_size, seq_len_k, d_model]。自注意力时与 query 同源；交叉注意力时为 Encoder 输出。
        mask : optional
            与注意力分数广播相容，例如 [batch_size, 1, seq_len_q, seq_len_k] 或
            [batch_size, num_heads, seq_len_q, seq_len_k]；bool 或 additive 均可（同 ScaledDotProductAttention）。

        返回
        ----
        output : [batch_size, seq_len_q, d_model]
        attn_weights : [batch_size, num_heads, seq_len_q, seq_len_k]
        """
        if key.size(1) != value.size(1):
            raise ValueError("key 与 value 的序列长度必须一致（均为 seq_len_k）")

        batch_size, seq_len_q, _ = query.shape
        seq_len_k = key.size(1)

        # 线性投影：[batch_size, seq_len_*, d_model]
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # 拆头：[batch_size, seq_len, d_model] -> [batch_size, seq_len, h, d_k] -> [batch_size, h, seq_len, d_k]
        Q = Q.view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        # 此时 Q: [batch_size, num_heads, seq_len_q, d_k]；K/V: [batch_size, num_heads, seq_len_k, d_k]

        attn_out, attn_weights = self.attention(Q, K, V, mask)
        # attn_out: [batch_size, num_heads, seq_len_q, d_k]

        # Concat(head_1,...,head_h)：[batch_size, seq_len_q, d_model]
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len_q, self.d_model)

        return self.W_o(attn_out), attn_weights

In [8]:
# 三种用法形状自检
B, d_model, h = 2, 512, 8
mha = MultiHeadAttention(d_model=d_model, num_heads=h, dropout=0.0)
Lq, Lk = 10, 20

# Encoder 自注意力：Q=K=V
x = torch.randn(B, Lk, d_model)
out_e, w_e = mha(x, x, x)
assert out_e.shape == (B, Lk, d_model) and w_e.shape == (B, h, Lk, Lk)

# Decoder 自注意力：同源序列 + 因果 mask（上三角为 True 表示禁止看未来）
y = torch.randn(B, Lq, d_model)
causal = torch.triu(torch.ones(Lq, Lq, dtype=torch.bool), diagonal=1).expand(B, h, Lq, Lq)
out_d, w_d = mha(y, y, y, mask=causal)
assert out_d.shape == (B, Lq, d_model) and w_d.shape == (B, h, Lq, Lq)

# Encoder-Decoder 交叉注意力：Q 来自解码端，K/V 来自编码端
q_dec = torch.randn(B, Lq, d_model)
mem = torch.randn(B, Lk, d_model)
out_x, w_x = mha(q_dec, mem, mem)
assert out_x.shape == (B, Lq, d_model) and w_x.shape == (B, h, Lq, Lk)

print("Encoder self:", out_e.shape, w_e.shape)
print("Decoder self (causal):", out_d.shape, w_d.shape)
print("Cross-attn:", out_x.shape, w_x.shape)

Encoder self: torch.Size([2, 20, 512]) torch.Size([2, 8, 20, 20])
Decoder self (causal): torch.Size([2, 10, 512]) torch.Size([2, 8, 10, 10])
Cross-attn: torch.Size([2, 10, 512]) torch.Size([2, 8, 10, 20])


**第三步：位置编码与前馈网络（Section 3.3 & 3.5）**

- **正弦位置编码（绝对位置）**：维 $2i$ 用 $\sin$，维 $2i+1$ 用 $\cos$：
  $$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right),\quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
  与词嵌入相加后通常再对和做 Dropout（与论文一致）。

- **按位置 FFN**：对每个位置独立地施加同一套 MLP：
  $$\mathrm{FFN}(x) = \max(0, xW_1+b_1)W_2+b_2$$
  隐层维度 $d_{ff}$ 默认 **2048**（与论文 base 模型一致）。

In [9]:
class PositionalEncoding(nn.Module):
    """
    正弦/余弦绝对位置编码（论文 Section 3.5）。
    前向：将 encodings 与输入 x 相加（x 一般为词嵌入），再对结果做 dropout。
    """

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # pe: [max_len, d_model]，每行对应一个 pos
        pe = torch.zeros(max_len, d_model)
        # position: [max_len, 1]，pos = 0, 1, ...
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        # 分母项 10000^(2i/d_model)，仅对偶数维索引 i 计算，形状约 [ceil(d_model/2)]
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # 注册为 buffer：不参与优化，但随模型 .to(device) 迁移
        self.register_buffer("pe", pe.unsqueeze(0))
        # pe 最终形状：[1, max_len, d_model]，便于广播到 batch

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : Tensor
            [batch_size, seq_len, d_model]（例如 token embedding）。
        返回
        ----
            同形状；内容为 x + PE(0:seq_len-1) 后经 dropout。
        """
        seq_len = x.size(1)
        # self.pe[:, :seq_len, :] -> [1, seq_len, d_model]，广播到整个 batch
        x = x + self.pe[:, :seq_len, :]
        return self.dropout(x)


class PositionwiseFeedForward(nn.Module):
    """
    按位置前馈网络（论文 Section 3.3）。
    FFN(x) = max(0, x W_1 + b_1) W_2 + b_2，对序列维逐位置共享参数。
    """

    def __init__(
        self,
        d_model: int,
        d_ff: int = 2048,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : [batch_size, seq_len, d_model]
        返回 : [batch_size, seq_len, d_model]
        """
        # 第一层 + ReLU：[batch_size, seq_len, d_ff]
        h = F.relu(self.w_1(x))
        h = self.dropout(h)
        # 第二层：[batch_size, seq_len, d_model]
        return self.w_2(h)

In [10]:
# 形状与公式自检
B, seq_len, d_model = 2, 16, 512
pe = PositionalEncoding(d_model=d_model, max_len=100, dropout=0.0)
pe.eval()
emb = torch.randn(B, seq_len, d_model)
out_pe = pe(emb)
assert out_pe.shape == emb.shape
assert torch.allclose(out_pe, emb + pe.pe[:, :seq_len, :])

ffn = PositionwiseFeedForward(d_model=d_model, d_ff=2048, dropout=0.0)
z = torch.randn(B, seq_len, d_model)
out_ffn = ffn(z)
assert out_ffn.shape == (B, seq_len, d_model)
manual_ffn = F.relu(z @ ffn.w_1.weight.T + ffn.w_1.bias) @ ffn.w_2.weight.T + ffn.w_2.bias
assert torch.allclose(out_ffn, manual_ffn, atol=1e-5)

print("PositionalEncoding:", out_pe.shape)
print("FFN:", out_ffn.shape, "d_ff=", ffn.w_1.out_features)

PositionalEncoding: torch.Size([2, 16, 512])
FFN: torch.Size([2, 16, 512]) d_ff= 2048


**第四步：Encoder / Decoder 层与堆叠（Post-LN）**

论文 **Post-LN**：每个子层先计算子层输出，对其做 **Dropout**，再与输入 **残差相加**，最后做 **LayerNorm**，即 $\mathrm{LayerNorm}(x+\mathrm{Dropout}(\mathrm{Sublayer}(x)))$。

- **EncoderLayer**：多头自注意力 + FFN，共两处 Post-LN。
- **DecoderLayer**：**掩码**自注意力 + **Encoder 记忆**上的交叉注意力 + FFN，共三处 Post-LN。
- **Encoder / Decoder**：`N` 层堆叠（默认 **$N=6$**），由 `nn.ModuleList` 实现。

In [11]:
class EncoderLayer(nn.Module):
    """
    单层 Encoder（论文 Fig.1 左侧）：Self-Attn + FFN，均为 Post-LN。
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # x: [batch_size, src_seq_len, d_model]
        attn_out, _ = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout1(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        return x


class DecoderLayer(nn.Module):
    """
    单层 Decoder：Masked Self-Attention + Encoder-Decoder Cross-Attn + FFN，均为 Post-LN。
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: Optional[torch.Tensor] = None,
        memory_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        # x: [batch_size, tgt_seq_len, d_model]；memory: [batch_size, src_seq_len, d_model]
        sa_out, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(sa_out))
        ca_out, _ = self.cross_attn(x, memory, memory, memory_mask)
        x = self.norm2(x + self.dropout2(ca_out))
        ff_out = self.ffn(x)
        x = self.norm3(x + self.dropout3(ff_out))
        return x


class Encoder(nn.Module):
    """N 层 EncoderLayer 堆叠（论文 base：N=6）。"""

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        N: int = 6,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(N)]
        )

    def forward(self, x: torch.Tensor, src_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, src_mask)
        return x


class Decoder(nn.Module):
    """N 层 DecoderLayer 堆叠（论文 base：N=6）。"""

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        N: int = 6,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(N)]
        )

    def forward(
        self,
        x: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: Optional[torch.Tensor] = None,
        memory_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, memory, tgt_mask, memory_mask)
        return x

In [12]:
# Encoder / Decoder 堆叠形状自检（N=2 以加快运行）
B, d_model, h, d_ff = 2, 512, 8, 2048
src_len, tgt_len = 20, 12
N_stack = 2

enc = Encoder(d_model=d_model, num_heads=h, d_ff=d_ff, N=N_stack, dropout=0.0)
dec = Decoder(d_model=d_model, num_heads=h, d_ff=d_ff, N=N_stack, dropout=0.0)
enc.eval()
dec.eval()

src = torch.randn(B, src_len, d_model)
tgt = torch.randn(B, tgt_len, d_model)
mem = enc(src)
causal = torch.triu(torch.ones(tgt_len, tgt_len, dtype=torch.bool), diagonal=1).expand(
    B, h, tgt_len, tgt_len
)
out = dec(tgt, mem, tgt_mask=causal)
assert mem.shape == (B, src_len, d_model) and out.shape == (B, tgt_len, d_model)
assert len(enc.layers) == N_stack == len(dec.layers)
print("Encoder layers:", len(enc.layers), "memory:", mem.shape, "Decoder out:", out.shape)

Encoder layers: 2 memory: torch.Size([2, 20, 512]) Decoder out: torch.Size([2, 12, 512])


**第五步：完整 Transformer、掩码与训练配套**

- **Embedding**：源/目标词嵌入 + $\sqrt{d_{\text{model}}}$ 缩放 + 正弦位置编码，再接 Encoder / Decoder 与输出投影 `generator`。
- **掩码**：`make_pad_mask` 处理 padding；`make_subsequent_mask` 生成解码器因果遮挡；`forward` 中交叉注意力使用 `src_mask` 屏蔽编码端 padding key。
- **损失**：Label Smoothing（论文 $\epsilon_{ls}=0.1$）。
- **优化**：Adam + **Noam** 预热调度 $lrate = d_{model}^{-0.5}\min(step^{-0.5},\, step\cdot warmup^{-1.5})$。

In [13]:
def make_pad_mask(seq: torch.Tensor, pad_idx: int) -> torch.Tensor:
    """
    构造 padding 注意力掩码。
    seq: [batch_size, seq_len]（token id）
    返回: [batch_size, 1, 1, seq_len]，True 表示该 **key 位置** 为 padding，应在注意力中屏蔽
         （与 ScaledDotProductAttention 中 bool mask 语义一致）。
    """
    return (seq == pad_idx).unsqueeze(1).unsqueeze(2)


def make_subsequent_mask(size: int, *, device: Optional[torch.device] = None) -> torch.Tensor:
    """
    解码器因果掩码（只看已生成前缀）。
    返回: [1, 1, size, size]，上三角（不含主对角）为 True，表示 query 不能注意未来的 key。
    """
    m = torch.triu(torch.ones(size, size, device=device, dtype=torch.bool), diagonal=1)
    return m.view(1, 1, size, size)


class Transformer(nn.Module):
    """
    论文 Fig.1：Embedding + PositionalEncoding + Encoder + Decoder + 线性输出层。
    """

    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model: int = 512,
        num_heads: int = 8,
        d_ff: int = 2048,
        N: int = 6,
        dropout: float = 0.1,
        max_len: int = 5000,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.src_embed = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len, dropout=dropout)
        self.encoder = Encoder(d_model, num_heads, d_ff, N=N, dropout=dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, N=N, dropout=dropout)
        self.generator = nn.Linear(d_model, tgt_vocab_size)

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src: torch.Tensor, src_mask: Optional[torch.Tensor]) -> torch.Tensor:
        # src: [batch_size, src_len]；src_mask: [batch_size, 1, 1, src_len] 等
        x = self.src_embed(src) * math.sqrt(self.d_model)
        x = self.pos_enc(x)
        return self.encoder(x, src_mask)

    def decode(
        self,
        trg: torch.Tensor,
        memory: torch.Tensor,
        src_mask: Optional[torch.Tensor],
        trg_mask: Optional[torch.Tensor],
    ) -> torch.Tensor:
        # trg: [batch_size, tgt_len]；memory: [batch_size, src_len, d_model]
        y = self.tgt_embed(trg) * math.sqrt(self.d_model)
        y = self.pos_enc(y)
        return self.decoder(y, memory, trg_mask, memory_mask=src_mask)

    def forward(
        self,
        src: torch.Tensor,
        trg: torch.Tensor,
        src_mask: Optional[torch.Tensor],
        trg_mask: Optional[torch.Tensor],
    ) -> torch.Tensor:
        """
        src / trg: [batch_size, src_len]、[batch_size, tgt_len]（整型 token id）
        src_mask: 一般为 make_pad_mask(src, pad_idx)，用于 Encoder 与 Decoder 交叉注意力
        trg_mask: 一般为 subsequent_mask | pad_mask(trg)，用于 Decoder 自注意力
        返回: logits [batch_size, tgt_len, tgt_vocab_size]
        """
        memory = self.encode(src, src_mask)
        dec_out = self.decode(trg, memory, src_mask, trg_mask)
        return self.generator(dec_out)

In [14]:
class LabelSmoothingLoss(nn.Module):
    """
    论文 Label Smoothing（ε=0.1）：非真实位置均匀分配 ε/(V-1)，真实位置为 1-ε；
    padding 目标位置在损失里 mask 掉（不参与分子分母）。
    """

    def __init__(self, vocab_size: int, padding_idx: int, smoothing: float = 0.1) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.padding_idx = padding_idx
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits: [batch, seq, vocab]，targets: [batch, seq]
        log_probs = F.log_softmax(logits, dim=-1)
        mask = targets.ne(self.padding_idx)
        with torch.no_grad():
            dist = torch.empty_like(log_probs).fill_(self.smoothing / (self.vocab_size - 1))
            dist.scatter_(-1, targets.unsqueeze(-1), 1.0 - self.smoothing)
        loss = F.kl_div(log_probs, dist, reduction="none").sum(-1)
        return (loss * mask.float()).sum() / mask.sum().clamp_min(1.0)


class NoamOpt:
    """
    Adam + Noam 调度（论文 5.4）：每步将 param group 的 lr 设为
    factor * d_model**(-0.5) * min(step**(-0.5), step * warmup**(-1.5))。
    用法：opt = Adam(..., lr=0)；包装器 noam = NoamOpt(model.d_model, 2, 4000, opt)；
    每个 batch：noam.step()（内部先按公式改 lr，再 optimizer.step()）。
    """

    def __init__(
        self,
        model_size: int,
        factor: float,
        warmup: int,
        optimizer: torch.optim.Optimizer,
    ) -> None:
        self.optimizer = optimizer
        self._step = 0
        self.warmup = warmup
        self.factor = factor
        self.model_size = model_size

    def step(self) -> None:
        self._step += 1
        lr = self.rate()
        for g in self.optimizer.param_groups:
            g["lr"] = lr
        self.optimizer.step()

    def rate(self, step: Optional[int] = None) -> float:
        if step is None:
            step = self._step
        step = max(1, step)
        return self.factor * (self.model_size ** (-0.5)) * min(
            step ** (-0.5), step * (self.warmup ** (-1.5))
        )

    def zero_grad(self) -> None:
        self.optimizer.zero_grad()


def build_noam_scheduler(
    optimizer: torch.optim.Optimizer,
    d_model: int,
    warmup_steps: int = 4000,
    factor: float = 1.0,
) -> torch.optim.lr_scheduler.LambdaLR:
    """
    与 Noam 公式等价的 LambdaLR，便于配合标准训练循环：
    loss.backward()；optimizer.step()；scheduler.step()。
    注意：LambdaLR 传入的 step 从 0 开始，这里用 step+1 对齐论文步计数。
    """

    def lr_lambda(step: int) -> float:
        step = step + 1
        return factor * (d_model ** (-0.5)) * min(step ** (-0.5), step * (warmup_steps ** (-1.5)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [ ]:
# 整模型前向与掩码、损失、调度器片段自检
pad = 0
src_vocab = tgt_vocab = 100
B, src_len, tgt_len = 2, 15, 11
model = Transformer(
    src_vocab, tgt_vocab, d_model=512, num_heads=8, d_ff=2048, N=2, dropout=0.0, max_len=128
)
model.eval()

src_ids = torch.randint(1, src_vocab, (B, src_len))
tgt_ids = torch.randint(1, tgt_vocab, (B, tgt_len))
src_ids[:, -3:] = pad
tgt_ids[:, -2:] = pad

src_m = make_pad_mask(src_ids, pad)
sub = make_subsequent_mask(tgt_len, device=tgt_ids.device)
trg_m = sub | make_pad_mask(tgt_ids, pad)

logits = model(src_ids, tgt_ids, src_m, trg_m)
assert logits.shape == (B, tgt_len, tgt_vocab)

crit = LabelSmoothingLoss(tgt_vocab, padding_idx=pad, smoothing=0.1)
gold = torch.randint(1, tgt_vocab, (B, tgt_len))
loss = crit(logits, gold)
assert loss.ndim == 0

inner = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
noam = NoamOpt(512, factor=2, warmup=4000, optimizer=inner)
assert noam.rate(1) > 0
sched = build_noam_scheduler(torch.optim.Adam(model.parameters(), lr=1e-3), d_model=512, warmup_steps=4000)
sched.step()
print("logits:", logits.shape, "loss:", float(loss), "LambdaLR last lr:", sched.get_last_lr()[0])

logits: torch.Size([2, 11, 100]) loss: 4.755154132843018 LambdaLR last lr: 3.493856214843422e-10


/home/hxy/miniconda3/envs/prml/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
